In [13]:
# !pip install scikit-optimize

### Logistic Regression

In [14]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### 1. 사전 작업


In [15]:
# ADASYN 적용된 학습 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")

print("shape:", train_df.shape)
print(train_df.head())
print("\ncolumns:")
print(train_df.columns.tolist())

# 실제 타깃 컬럼명
target_col = "Risk_Label"

# 전체 데이터(X)와 타깃(y) 분리
# 주의: X에 Risk_Label이 포함되면 데이터 누수가 생기므로 반드시 제거
X = train_df.drop(columns=[target_col]).copy()
y = train_df[target_col].astype(int)

print("\nX shape:", X.shape)
print("y distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

# 불균형 데이터 평가를 위한 G-Mean 함수
def gmean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return np.sqrt(recall * specificity)


shape: (2466, 10)
   NASDAQ_return(%)  Brent Crude Oil_return(%)  Gold Spot_return(%)  \
0          0.593612                   0.545323             0.488488   
1          0.259590                   0.545323             0.696235   
2          0.758918                   0.545323             0.535190   
3          0.592042                   0.545323             0.630347   
4          0.610832                   0.545323             0.657784   

   KOSDAQ_return(%)  KOSPI 200 Volume  KOSPI 200 lagged_1_return(%)  \
0          0.349814          0.000817                      0.593898   
1          0.617084          0.000552                      0.548072   
2          0.580751          0.000633                      0.616220   
3          0.668308          0.000972                      0.551119   
4          0.566195          0.000917                      0.688352   

   KOSPI 200_OG  KOSPI 200_PPO  VKOSPI_Change(%)  Risk_Label  
0      0.728036       1.000000          0.298311           0  
1 

### 2. Valid 기반 파라미터 최적화

In [16]:
# =========================
# 2. Valid 기반 LR 하이퍼파라미터 + cutoff 동시 탐색
# =========================
import os
import numpy as np
import pandas as pd
import sklearn
from packaging import version

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    log_loss, roc_auc_score, average_precision_score
)

# -------------------------
# 1) 데이터 로드
# -------------------------
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")

target_col = "Risk_Label"

# X에 target이 들어가면 데이터 누수이므로 반드시 제거
X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].astype(int)

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid = valid_df[target_col].astype(int)

# valid/test에서 컬럼 불일치가 있으면 0으로 채우지 말고 오류 내는 게 안전함
missing_in_valid = set(X_train_raw.columns) - set(X_valid_raw.columns)
extra_in_valid = set(X_valid_raw.columns) - set(X_train_raw.columns)

if missing_in_valid or extra_in_valid:
    raise ValueError(
        f"train-valid 컬럼 불일치\n"
        f"valid에 없는 train 컬럼: {missing_in_valid}\n"
        f"valid에만 있는 컬럼: {extra_in_valid}"
    )

X_valid_raw = X_valid_raw[X_train_raw.columns]

X_train_model = X_train_raw.values
X_valid_model = X_valid_raw.values
feature_names = X_train_raw.columns

print("Train shape:", X_train_model.shape)
print("Valid shape:", X_valid_model.shape)
print("\nTrain class distribution")
print(y_train.value_counts(normalize=True))
print("\nValid class distribution")
print(y_valid.value_counts(normalize=True))


# -------------------------
# 2) 평가 함수
# -------------------------
def get_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = np.sqrt(rec * specificity)
    
    # H1 스코어: F1과 G-Mean의 조화 평균
    h1 = float(2 * f1 * gmean / (f1 + gmean)) if (f1 + gmean) > 0 else 0.0

    try:
        ll = log_loss(y_true, y_prob, labels=[0, 1])
    except Exception:
        ll = np.nan

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except Exception:
        pr_auc = np.nan

    return {
        "threshold": threshold,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "specificity": specificity,
        "f1": f1,
        "gmean": gmean,
        "h1": h1,
        "log_loss": ll,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def make_threshold_grid(valid_prob):
    # 0.01 간격 cutoff + validation 확률 분위수 기반 cutoff를 함께 사용
    # 확률이 특정 구간에 몰려 있어도 후보 cutoff가 충분히 잡히도록 함
    return np.unique(np.r_[
        np.arange(0.01, 0.991, 0.01),
        np.quantile(valid_prob, np.linspace(0.01, 0.99, 99))
    ])


def find_best_cutoff(y_valid, valid_prob):
    threshold_history = []

    for thr in make_threshold_grid(valid_prob):
        m = get_binary_metrics(y_valid, valid_prob, threshold=thr)
        threshold_history.append({
            "threshold": float(thr),
            "valid_accuracy": m["accuracy"],
            "valid_precision": m["precision"],
            "valid_recall": m["recall"],
            "valid_specificity": m["specificity"],
            "valid_f1": m["f1"],
            "valid_gmean": m["gmean"],
            "valid_h1": m["h1"],
            "valid_log_loss": m["log_loss"],
            "valid_roc_auc": m["roc_auc"],
            "valid_pr_auc": m["pr_auc"],
            "tn": m["tn"],
            "fp": m["fp"],
            "fn": m["fn"],
            "tp": m["tp"]
        })

    threshold_df = pd.DataFrame(threshold_history)

    # cutoff 선택 기준:
    # 1순위 H1 스코어 최대 (F1과 G-Mean의 조화 평균)
    # 2순위 G-Mean 최대
    # 3순위 F1 최대
    threshold_df = threshold_df.sort_values(
        by=["valid_h1", "valid_gmean", "valid_f1"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    return threshold_df.iloc[0], threshold_df


# -------------------------
# 3) sklearn 버전 호환 LR 생성 함수
# -------------------------
def make_lr(C, l1_ratio, max_iter=5000, random_state=1):
    kwargs = dict(
        solver="saga",
        C=C,
        l1_ratio=l1_ratio,
        max_iter=max_iter,
        random_state=random_state,
        tol=1e-4
    )

    # sklearn 1.8부터 penalty 경고가 뜨므로 버전별 처리
    if version.parse(sklearn.__version__) < version.parse("1.8"):
        kwargs["penalty"] = "elasticnet"

    return LogisticRegression(**kwargs)


# -------------------------
# 4) 1차 coarse grid search
#    C, l1_ratio, cutoff를 valid set에서 동시에 선택
# -------------------------
C_grid_1 = np.logspace(-4, 4, 33)
l1_grid_1 = np.round(np.linspace(0.0, 1.0, 21), 2)

search_history_1 = []
threshold_history_all_1 = []

print(f"\n[1차 탐색] LR 조합 수: {len(C_grid_1) * len(l1_grid_1)}")
print("각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.")

for C_val in C_grid_1:
    for l1_val in l1_grid_1:
        model_trial = make_lr(C=C_val, l1_ratio=l1_val)
        model_trial.fit(X_train_model, y_train)

        valid_prob = model_trial.predict_proba(X_valid_model)[:, 1]
        best_thr_row, threshold_df_tmp = find_best_cutoff(y_valid, valid_prob)

        search_history_1.append({
            "stage": "coarse",
            "C": C_val,
            "l1_ratio": l1_val,
            "best_cutoff": best_thr_row["threshold"],
            "valid_accuracy": best_thr_row["valid_accuracy"],
            "valid_precision": best_thr_row["valid_precision"],
            "valid_recall": best_thr_row["valid_recall"],
            "valid_specificity": best_thr_row["valid_specificity"],
            "valid_f1": best_thr_row["valid_f1"],
            "valid_gmean": best_thr_row["valid_gmean"],
            "valid_h1": best_thr_row["valid_h1"],
            "valid_log_loss": best_thr_row["valid_log_loss"],
            "valid_roc_auc": best_thr_row["valid_roc_auc"],
            "valid_pr_auc": best_thr_row["valid_pr_auc"],
            "tn": best_thr_row["tn"],
            "fp": best_thr_row["fp"],
            "fn": best_thr_row["fn"],
            "tp": best_thr_row["tp"]
        })

        threshold_df_tmp = threshold_df_tmp.copy()
        threshold_df_tmp["stage"] = "coarse"
        threshold_df_tmp["C"] = C_val
        threshold_df_tmp["l1_ratio"] = l1_val
        threshold_history_all_1.append(threshold_df_tmp)

search_df_1 = pd.DataFrame(search_history_1)

search_df_1 = search_df_1.sort_values(
    by=["valid_h1", "valid_gmean", "valid_f1", "valid_log_loss"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n[1차 탐색 상위 10개]")
print(search_df_1.head(10))

best_1 = search_df_1.iloc[0]
best_C_1 = float(best_1["C"])
best_l1_1 = float(best_1["l1_ratio"])

print("\n[1차 Best]")
print(f"C={best_C_1:.8f}, l1_ratio={best_l1_1:.4f}, cutoff={best_1['best_cutoff']:.4f}, "
      f"H1={best_1['valid_h1']:.4f}, G-Mean={best_1['valid_gmean']:.4f}, F1={best_1['valid_f1']:.4f}, "
      f"Recall={best_1['valid_recall']:.4f}")

Train shape: (2466, 9)
Valid shape: (1438, 9)

Train class distribution
Risk_Label
0    0.675182
1    0.324818
Name: proportion, dtype: float64

Valid class distribution
Risk_Label
0    0.887344
1    0.112656
Name: proportion, dtype: float64

[1차 탐색] LR 조합 수: 693
각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.

[1차 탐색 상위 10개]
    stage         C  l1_ratio  best_cutoff  valid_accuracy  valid_precision  \
0  coarse  1.000000      0.00     0.389166        0.799722         0.281250   
1  coarse  1.000000      0.25     0.390000        0.797636         0.278351   
2  coarse  0.562341      0.40     0.390000        0.800417         0.280702   
3  coarse  1.778279      0.10     0.390000        0.794159         0.275168   
4  coarse  1.000000      0.55     0.390000        0.794159         0.275168   
5  coarse  1.000000      0.30     0.390000        0.796940         0.277397   
6  coarse  1.778279      0.05     0.390000        0.793463         0.274247   
7  coarse  1.000000      0.35     0.390000      

In [17]:
# -------------------------
# 5) 2차 local refinement
#    1차 best 주변에서 C, l1_ratio, cutoff 동시 재탐색
# -------------------------
C_grid_2 = best_C_1 * np.logspace(-0.5, 0.5, 15)
l1_min = max(0.0, best_l1_1 - 0.20)
l1_max = min(1.0, best_l1_1 + 0.20)
l1_grid_2 = np.round(np.linspace(l1_min, l1_max, 15), 3)

search_history_2 = []
threshold_history_all_2 = []

print(f"\n[2차 세부 탐색] LR 조합 수: {len(C_grid_2) * len(l1_grid_2)}")
print("각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.")

for C_val in C_grid_2:
    for l1_val in l1_grid_2:
        model_trial = make_lr(C=C_val, max_iter=5000, l1_ratio=l1_val)
        model_trial.fit(X_train_model, y_train)

        valid_prob = model_trial.predict_proba(X_valid_model)[:, 1]
        best_thr_row, threshold_df_tmp = find_best_cutoff(y_valid, valid_prob)

        search_history_2.append({
            "stage": "refine",
            "C": C_val,
            "l1_ratio": l1_val,
            "best_cutoff": best_thr_row["threshold"],
            "valid_accuracy": best_thr_row["valid_accuracy"],
            "valid_precision": best_thr_row["valid_precision"],
            "valid_recall": best_thr_row["valid_recall"],
            "valid_specificity": best_thr_row["valid_specificity"],
            "valid_f1": best_thr_row["valid_f1"],
            "valid_gmean": best_thr_row["valid_gmean"],
            "valid_h1": best_thr_row["valid_h1"],
            "valid_log_loss": best_thr_row["valid_log_loss"],
            "valid_roc_auc": best_thr_row["valid_roc_auc"],
            "valid_pr_auc": best_thr_row["valid_pr_auc"],
            "tn": best_thr_row["tn"],
            "fp": best_thr_row["fp"],
            "fn": best_thr_row["fn"],
            "tp": best_thr_row["tp"]
        })

        threshold_df_tmp = threshold_df_tmp.copy()
        threshold_df_tmp["stage"] = "refine"
        threshold_df_tmp["C"] = C_val
        threshold_df_tmp["l1_ratio"] = l1_val
        threshold_history_all_2.append(threshold_df_tmp)

search_df_2 = pd.DataFrame(search_history_2)

search_df = pd.concat([search_df_1, search_df_2], ignore_index=True)

search_df = search_df.sort_values(
    by=["valid_h1", "valid_gmean", "valid_f1", "valid_log_loss"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

best_row = search_df.iloc[0]
best_lr_params = {
    "C": float(best_row["C"]),
    "l1_ratio": float(best_row["l1_ratio"])
}
best_cutoff = float(best_row["best_cutoff"])

print("\n[최종 Best Hyperparameters + Cutoff]")
print(f"C        : {best_lr_params['C']:.8f}")
print(f"l1_ratio : {best_lr_params['l1_ratio']:.4f}")
print(f"cutoff   : {best_cutoff:.4f}")
print(f"H1-Score : {best_row['valid_h1']:.4f}")
print(f"G-Mean   : {best_row['valid_gmean']:.4f}")
print(f"F1       : {best_row['valid_f1']:.4f}")
print(f"Recall   : {best_row['valid_recall']:.4f}")
print(f"Specificity: {best_row['valid_specificity']:.4f}")
print(f"Precision: {best_row['valid_precision']:.4f}")
print(f"LogLoss  : {best_row['valid_log_loss']:.4f}")

print("\n[전체 탐색 상위 15개]")
print(search_df.head(15))



# -------------------------
# 6) 최종 모델 학습
# -------------------------
model = make_lr(
    C=best_lr_params["C"],
    l1_ratio=best_lr_params["l1_ratio"]
)
model.fit(X_train_model, y_train)


[2차 세부 탐색] LR 조합 수: 225
각 LR 조합마다 valid set에서 cutoff를 함께 탐색합니다.

[최종 Best Hyperparameters + Cutoff]
C        : 1.00000000
l1_ratio : 0.0000
cutoff   : 0.3892
H1-Score : 0.4627
G-Mean   : 0.6472
F1       : 0.3600
Recall   : 0.5000
Specificity: 0.8378
Precision: 0.2812
LogLoss  : 0.4374

[전체 탐색 상위 15개]
     stage         C  l1_ratio  best_cutoff  valid_accuracy  valid_precision  \
0   coarse  1.000000     0.000     0.389166        0.799722         0.281250   
1   refine  1.000000     0.000     0.389166        0.799722         0.281250   
2   refine  1.178769     0.000     0.390000        0.798331         0.279310   
3   refine  1.178769     0.014     0.390000        0.798331         0.279310   
4   refine  1.178769     0.029     0.390000        0.798331         0.279310   
5   refine  1.178769     0.043     0.390000        0.798331         0.279310   
6   refine  1.178769     0.057     0.390000        0.797636         0.278351   
7   refine  1.178769     0.071     0.390000        0.7976

LogisticRegression(l1_ratio=0.0, max_iter=5000, penalty='elasticnet',
                   random_state=1, solver='saga')

### 3. test data 성능 평가


In [19]:
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

X_test_df = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].astype(int)

missing_in_test = set(feature_names) - set(X_test_df.columns)
extra_in_test = set(X_test_df.columns) - set(feature_names)

if missing_in_test or extra_in_test:
    raise ValueError(
        f"train-test 컬럼 불일치\n"
        f"test에 없는 train 컬럼: {missing_in_test}\n"
        f"test에만 있는 컬럼: {extra_in_test}"
    )

X_test_df = X_test_df[feature_names]
X_test = X_test_df.values

# Test 예측
y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_cutoff).astype(int)
test_metrics = get_binary_metrics(y_test, y_test_prob, threshold=best_cutoff)

# Valid 재계산 (Cell 8에서 로드된 valid_df 사용)
y_valid_prob = model.predict_proba(X_valid_model)[:, 1]
y_valid_pred = (y_valid_prob >= best_cutoff).astype(int)
valid_metrics = get_binary_metrics(y_valid, y_valid_prob, threshold=best_cutoff)

# ========== 성능 출력 ==========
print("\n[VALID performance]")
print(f"Cutoff    : {best_cutoff:.4f}")
print(f"Accuracy  : {valid_metrics['accuracy']:.4f}")
print(f"Precision : {valid_metrics['precision']:.4f}")
print(f"Recall    : {valid_metrics['recall']:.4f}")
print(f"Specificity: {valid_metrics['specificity']:.4f}")
print(f"F1-score  : {valid_metrics['f1']:.4f}")
print(f"G-Mean    : {valid_metrics['gmean']:.4f}")
print(f"H1-Score  : {valid_metrics['h1']:.4f}")
print(f"ROC-AUC   : {valid_metrics['roc_auc']:.4f}")
print(f"PR-AUC    : {valid_metrics['pr_auc']:.4f}")
print(f"LogLoss   : {valid_metrics['log_loss']:.4f}")

print("\n[TEST performance]")
print(f"Cutoff    : {best_cutoff:.4f}")
print(f"Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"Precision : {test_metrics['precision']:.4f}")
print(f"Recall    : {test_metrics['recall']:.4f}")
print(f"Specificity: {test_metrics['specificity']:.4f}")
print(f"F1-score  : {test_metrics['f1']:.4f}")
print(f"G-Mean    : {test_metrics['gmean']:.4f}")
print(f"H1-Score  : {test_metrics['h1']:.4f}")
print(f"ROC-AUC   : {test_metrics['roc_auc']:.4f}")
print(f"PR-AUC    : {test_metrics['pr_auc']:.4f}")
print(f"LogLoss   : {test_metrics['log_loss']:.4f}")

print("\n[TEST Confusion Matrix]")
print(confusion_matrix(y_test, y_test_pred, labels=[0, 1]))

print("\n[TEST Classification Report]")
print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))

# ========== 결과 저장 ==========
valid_pred_df = valid_df.copy()
valid_pred_df["LR_Prob"] = y_valid_prob
valid_pred_df["LR_Pred"] = y_valid_pred

test_pred_df = test_df.copy()
test_pred_df["LR_Prob"] = y_test_prob
test_pred_df["LR_Pred"] = y_test_pred

valid_pred_df.to_csv("../../data/processed/result/LR_valid_pred.csv", index=False)
test_pred_df.to_csv("../../data/processed/result/LR_test_pred.csv", index=False)
print("\n✓ \"LR_valid_pred.csv\", \"LR_test_pred.csv\" 파일로 저장 완료")


[VALID performance]
Cutoff    : 0.3291
Accuracy  : 0.7031
Precision : 0.2088
Recall    : 0.5864
Specificity: 0.7179
F1-score  : 0.3079
G-Mean    : 0.6488
H1-Score  : 0.4177
ROC-AUC   : 0.7035
PR-AUC    : 0.3057
LogLoss   : 0.4471

[TEST performance]
Cutoff    : 0.3291
Accuracy  : 0.7299
Precision : 0.2213
Recall    : 0.5714
Specificity: 0.7497
F1-score  : 0.3190
G-Mean    : 0.6545
H1-Score  : 0.4290
ROC-AUC   : 0.7239
PR-AUC    : 0.2747
LogLoss   : 0.4338

[TEST Confusion Matrix]
[[548 183]
 [ 39  52]]

[TEST Classification Report]
              precision    recall  f1-score   support

           0     0.9336    0.7497    0.8316       731
           1     0.2213    0.5714    0.3190        91

    accuracy                         0.7299       822
   macro avg     0.5774    0.6605    0.5753       822
weighted avg     0.8547    0.7299    0.7748       822


✓ "LR_valid_pred.csv", "LR_test_pred.csv" 파일로 저장 완료


In [23]:
print("\n[TEST Confusion Matrix]")
display(confusion_matrix(y_valid, y_valid_pred, labels=[0, 1]))

print("\n[TEST Confusion Matrix]")
display(confusion_matrix(y_test, y_test_pred, labels=[0, 1]))




[TEST Confusion Matrix]


array([[916, 360],
       [ 67,  95]], dtype=int64)


[TEST Confusion Matrix]


array([[548, 183],
       [ 39,  52]], dtype=int64)